In [1]:
import os
import math
import numpy as np
import pandas as pd

# =========================================================
# PATHS / CONFIG
# =========================================================
INPUT_DIR  = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_input_files"
OUTPUT_DIR = r"C:\Users\loci_\Desktop\trading_webapp\DATA"

RX_FILE = os.path.join(INPUT_DIR, "RX1_small.csv")
AD_FILE = os.path.join(INPUT_DIR, "AD1_small.csv")

START_NAV_USD   = 10_000_000
ANNUAL_VOL      = 0.20
TRADING_DAYS    = 256
VOL_LOOKBACK    = 36
DECAY           = 2 / (VOL_LOOKBACK + 1)   # 2/37

# Fixed PDM you asked for
PDM_CONST       = 1.6

# Instrument weights for portfolio (used for portfolio metrics)
WEIGHTS = {"RX1": 0.5, "AD1": 0.5}

# EWMA family: slow = 4 × fast, with your scalers
EWMA_SETUPS = [
    {"fast": 2,  "slow":  8,  "scaler": 10.6},
    {"fast": 4,  "slow": 16,  "scaler":  7.5},
    {"fast": 8,  "slow": 32,  "scaler":  5.3},
    {"fast": 16, "slow": 64,  "scaler":  3.75},
    {"fast": 32, "slow":128,  "scaler":  2.65},
    {"fast": 64, "slow":256,  "scaler":  1.87},
]

# =========================================================
# METRICS HELPERS
# =========================================================
def _annualize_mean(mu_daily, trading_days=TRADING_DAYS):
    return mu_daily * trading_days

def _annualize_vol(sig_daily, trading_days=TRADING_DAYS):
    return sig_daily * np.sqrt(trading_days)

def _downside_std(returns):
    r = pd.Series(returns).dropna()
    r_dn = r[r < 0]
    if len(r_dn) < 2:
        return np.nan
    return r_dn.std(ddof=0)

def _max_drawdown(returns):
    r = pd.Series(returns).fillna(0.0)
    nav = (1.0 + r).cumprod()
    peak = nav.cummax()
    dd = (nav / peak) - 1.0
    return dd.min(), dd, nav

def performance_metrics(returns, trading_days=TRADING_DAYS):
    s = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3:
        return pd.Series({
            "ann_return": np.nan, "ann_vol": np.nan,
            "sharpe": np.nan, "sortino": np.nan,
            "max_dd": np.nan, "calmar": np.nan,
            "hit_rate": np.nan, "skew": np.nan, "kurt": np.nan
        })

    mu_d   = s.mean()
    sig_d  = s.std(ddof=0)
    dn_std = _downside_std(s)

    ann_ret = _annualize_mean(mu_d, trading_days)
    ann_vol = _annualize_vol(sig_d, trading_days)
    sharpe  = np.nan if sig_d == 0 else (mu_d / sig_d) * np.sqrt(trading_days)
    sortino = np.nan if (pd.isna(dn_std) or dn_std == 0) else (mu_d / dn_std) * np.sqrt(trading_days)

    mdd, _, _ = _max_drawdown(s)
    calmar = np.nan if (pd.isna(mdd) or mdd == 0) else (ann_ret / abs(mdd))

    hit_rate = (s > 0).mean()
    skew = s.skew()
    kurt = s.kurtosis()

    return pd.Series({
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_dd": mdd,
        "calmar": calmar,
        "hit_rate": hit_rate,
        "skew": skew,
        "kurt": kurt
    })

def collect_metrics_for_df(df, label):
    m_exec = performance_metrics(df["strategy_ret"])
    m_raw  = performance_metrics(df["forecast_x_return"])
    out = pd.concat([m_raw.add_prefix("raw_"), m_exec.add_prefix("exec_")])
    out.name = label
    return out

def portfolio_metrics(rx_df, ad_df, w_rx=0.5, w_ad=0.5):
    port_exec = (w_rx * rx_df["strategy_ret"].fillna(0.0) +
                 w_ad * ad_df["strategy_ret"].fillna(0.0))
    port_raw  = (w_rx * rx_df["forecast_x_return"].fillna(0.0) +
                 w_ad * ad_df["forecast_x_return"].fillna(0.0))
    m_exec = performance_metrics(port_exec)
    m_raw  = performance_metrics(port_raw)
    return pd.concat([m_raw.add_prefix("raw_"), m_exec.add_prefix("exec_")])

# =========================================================
# ONE-INSTRUMENT / ONE-EWMA BACKTEST
# =========================================================
def run_one_ewma(df_raw, instr_name, fast, slow, scaler, weight, pdm, output_dir):
    """
    df_raw columns expected:
      Date, PX_CLOSE_1D, st_dev, Exchange rate
      (optional) TICK_VALUE, TICK_SIZE, POINT_VALUE
    """
    df = df_raw.copy()
    df = df.dropna(subset=["Date"]).sort_values("Date").set_index("Date")

    price = df["PX_CLOSE_1D"].astype(float)
    stdev_price = df["st_dev"].astype(float)

    # FX: you specified IVV should be lower than ICV, meaning FX column is EUR per USD (>1)
    # So: EUR → USD = divide by FX
    if "Exchange rate" in df.columns:
        fx_series = df["Exchange rate"].astype(float)
    else:
        fx_series = pd.Series(1.0, index=df.index)

    # Tick / point info
    if "TICK_VALUE" in df.columns and "TICK_SIZE" in df.columns:
        tick_value = float(df["TICK_VALUE"].iloc[0])
        tick_size  = float(df["TICK_SIZE"].iloc[0])
        MULT = tick_value / tick_size
    else:
        tick_value = 1000.0
        tick_size  = 0.01
        MULT = tick_value / tick_size

    # 1) returns for signal & variance
    df["ret_pct_for_sig"] = price.pct_change()
    df["ret_net_for_var"] = price.diff()

    # 2) EWMA variance (squared net returns)
    variance = [np.nan] * len(df)
    ret_net_vals = df["ret_net_for_var"].values
    first_idx = np.where(~np.isnan(ret_net_vals))[0][0]
    first_sq = ret_net_vals[first_idx] ** 2
    variance[first_idx] = first_sq
    prev_var = first_sq
    for i in range(first_idx + 1, len(df)):
        rn = ret_net_vals[i]
        rn2 = 0.0 if np.isnan(rn) else rn * rn
        new_var = DECAY * rn2 + (1 - DECAY) * prev_var
        variance[i] = new_var
        prev_var = new_var
    df["variance"] = variance
    df["stdev_variance"] = np.sqrt(df["variance"])

    # 3) EWMAs for signal
    df["ewma_fast"] = price.ewm(span=fast, adjust=False).mean()
    df["ewma_slow"] = price.ewm(span=slow,  adjust=False).mean()
    df["raw_crossover"] = df["ewma_fast"] - df["ewma_slow"]
    df["vol_adj_crossover"] = df["raw_crossover"] / df["stdev_variance"]

    # 4) Forecast (with family scaler) & cap
    df["scaled_forecast"] = df["vol_adj_crossover"] * scaler
    df["capped_forecast"] = df["scaled_forecast"].clip(-20, 20)

    # Raw signal PnL proxy
    df["forecast_x_return"] = df["capped_forecast"].shift(1) * df["ret_pct_for_sig"]

    # 5) Cash vol chain
    df["price_volatility"] = (stdev_price / price * 100).round(2)
    df["one_pct_move"]     = price * 0.01
    # Use point value if present, else MULT
    if "POINT_VALUE" in df.columns:
        point_value = float(df["POINT_VALUE"].iloc[0])
    else:
        point_value = MULT
    df["block_value_eur"]  = df["one_pct_move"] * point_value
    df["icv_eur"]          = df["price_volatility"] * df["block_value_eur"]
    # *** KEY: IVV = ICV (EUR) / FX (EUR per USD) → IVV < ICV, in USD
    df["ivv"]              = df["icv_eur"] / fx_series

    # 6) Iterative sizing with lagged NAV
    nav_list = []
    dcv_list = []                 # daily_cash_vol_target
    target_contracts_list = []
    trades_list = []
    current_pos_list = []
    strat_ret_list = []
    vol_scaler_list = []
    subsystem_pos_list = []
    port_instr_pos_list = []

    prev_nav = START_NAV_USD
    prev_dcv = prev_nav * ANNUAL_VOL / math.sqrt(TRADING_DAYS)
    prev_target = 0

    for i, (date, row) in enumerate(df.iterrows()):
        px_today = row["PX_CLOSE_1D"]
        if i == 0:
            delta_price = 0.0
        else:
            px_yday = df["PX_CLOSE_1D"].iloc[i - 1]
            delta_price = px_today - px_yday

        # 1) risk budget = yesterday's NAV
        dcv_today = prev_dcv

        # 2) vol scaler
        ivv_today = row["ivv"]
        if np.isnan(ivv_today) or ivv_today == 0:
            vol_scaler_today = np.nan
        else:
            vol_scaler_today = dcv_today / ivv_today

        # 3) subsystem position
        cap_f = row["capped_forecast"]
        subsystem_pos = 0.0 if (np.isnan(cap_f) or np.isnan(vol_scaler_today)) else (cap_f * vol_scaler_today) / 10.0

        # 4) portfolio instrument position BEFORE rounding
        port_instr_pos = subsystem_pos * weight * pdm

        # 5) target contracts (rounded)
        target_contracts = int(round(port_instr_pos))
        trades = target_contracts - prev_target

        # 6) carry PnL only (no exec prices yet)
        carry_pnl_eur = prev_target * delta_price * MULT
        fx_today = fx_series.loc[date]
        carry_pnl_usd = carry_pnl_eur / fx_today   # EUR → USD (divide) because FX is EUR per USD

        pnl_usd = carry_pnl_usd
        nav_today = prev_nav + pnl_usd
        strat_ret = 0.0 if prev_nav == 0 else (pnl_usd / prev_nav)

        # store
        nav_list.append(nav_today)
        dcv_list.append(dcv_today)
        target_contracts_list.append(target_contracts)
        trades_list.append(trades)
        current_pos_list.append(target_contracts)
        strat_ret_list.append(strat_ret)
        vol_scaler_list.append(vol_scaler_today)
        subsystem_pos_list.append(subsystem_pos)
        port_instr_pos_list.append(port_instr_pos)

        # roll
        prev_nav = nav_today
        prev_dcv = nav_today * ANNUAL_VOL / math.sqrt(TRADING_DAYS)
        prev_target = target_contracts

    # 7) Assign back
    df["nav_usd"]                 = nav_list
    df["daily_cash_vol_target"]   = dcv_list
    df["volatility_scaler"]       = vol_scaler_list
    df["subsystem_position"]      = subsystem_pos_list
    df["portfolio_instr_position"]= port_instr_pos_list
    df["target_contracts"]        = target_contracts_list
    df["current_position"]        = current_pos_list
    df["trades"]                  = trades_list
    df["strategy_ret"]            = strat_ret_list
    df["cumulative_pnl_usd"]      = df["nav_usd"] - START_NAV_USD
    df["pdm_used"]                = pdm
    df["eff_multiplier"]          = weight * pdm

    # 8) Metrics & turnover
    raw_sharpe  = performance_metrics(df["forecast_x_return"])["sharpe"]
    exec_sharpe = performance_metrics(df["strategy_ret"])["sharpe"]

    # turnover
    years = len(df) / TRADING_DAYS if len(df) else np.nan
    avg_lots_traded_yearly = df["trades"].abs().sum() / years if years and years > 0 else np.nan
    avg_abs_pos            = df["current_position"].abs().mean()
    turnover = (avg_lots_traded_yearly / (2 * avg_abs_pos)) if (avg_abs_pos and avg_abs_pos != 0) else np.nan

    # 9) Save per-instrument CSV for this EWMA setup
    out_name = f"{instr_name}_ewma{fast}_{slow}.csv"
    out_path = os.path.join(output_dir, out_name)
    df.reset_index().to_csv(out_path, index=False)

    return {
        "df": df,
        "instrument": instr_name,
        "fast": fast, "slow": slow, "scaler": scaler,
        "raw_sharpe": raw_sharpe,
        "exec_sharpe": exec_sharpe,
        "turnover": turnover,
        "avg_yearly_lots": avg_lots_traded_yearly,
        "avg_abs_pos": avg_abs_pos,
        "csv": out_path
    }

# =========================================================
# LOAD INPUT DATA
# =========================================================
rx_raw = pd.read_csv(RX_FILE)
ad_raw = pd.read_csv(AD_FILE)

# Ensure Date parsing (day-first)
rx_raw["Date"] = pd.to_datetime(rx_raw["Date"], dayfirst=True, errors="coerce")
ad_raw["Date"] = pd.to_datetime(ad_raw["Date"], dayfirst=True, errors="coerce")

rx_raw = rx_raw.dropna(subset=["Date"]).sort_values("Date")
ad_raw = ad_raw.dropna(subset=["Date"]).sort_values("Date")

# =========================================================
# RUN SWEEP
# =========================================================
summary_rows = []

for setup in EWMA_SETUPS:
    f, s, sc = setup["fast"], setup["slow"], setup["scaler"]
    print(f"\n=== EWMA {f}/{s} (scaler={sc}) ===")

    rx_res = run_one_ewma(rx_raw, "RX1", f, s, sc, WEIGHTS["RX1"], PDM_CONST, OUTPUT_DIR)
    ad_res = run_one_ewma(ad_raw, "AD1", f, s, sc, WEIGHTS["AD1"], PDM_CONST, OUTPUT_DIR)

    rx_df = rx_res["df"]; ad_df = ad_res["df"]

    # instrument metrics (rich)
    rx_metrics = collect_metrics_for_df(rx_df, f"RX1_{f}/{s}")
    ad_metrics = collect_metrics_for_df(ad_df, f"AD1_{f}/{s}")

    # portfolio 50/50 metrics for this speed
    port_metrics = portfolio_metrics(rx_df, ad_df, w_rx=WEIGHTS["RX1"], w_ad=WEIGHTS["AD1"])

    # Console prints (core)
    print(f"RX1  raw Sharpe={rx_metrics['raw_sharpe']:.3f},  Sortino={rx_metrics['raw_sortino']:.3f},  MDD={rx_metrics['raw_max_dd']:.2%}")
    print(f"RX1  exec Sharpe={rx_metrics['exec_sharpe']:.3f}, Sortino={rx_metrics['exec_sortino']:.3f}, MDD={rx_metrics['exec_max_dd']:.2%}")
    print(f"AD1  raw Sharpe={ad_metrics['raw_sharpe']:.3f},  Sortino={ad_metrics['raw_sortino']:.3f},  MDD={ad_metrics['raw_max_dd']:.2%}")
    print(f"AD1  exec Sharpe={ad_metrics['exec_sharpe']:.3f}, Sortino={ad_metrics['exec_sortino']:.3f}, MDD={ad_metrics['exec_max_dd']:.2%}")
    print(f"PORT raw Sharpe={port_metrics['raw_sharpe']:.3f},  Sortino={port_metrics['raw_sortino']:.3f},  MDD={port_metrics['raw_max_dd']:.2%}")
    print(f"PORT exec Sharpe={port_metrics['exec_sharpe']:.3f}, Sortino={port_metrics['exec_sortino']:.3f}, MDD={port_metrics['exec_max_dd']:.2%}")

    # collect summary rows (instrument + portfolio, raw & exec)
    for (label, m) in [("RX1", rx_metrics), ("AD1", ad_metrics), ("PORT", port_metrics)]:
        for kind in ["raw_", "exec_"]:
            row = {
                "instrument": label,
                "fast": f, "slow": s, "scaler": sc,
                "ann_return": m[kind + "ann_return"],
                "ann_vol":    m[kind + "ann_vol"],
                "sharpe":     m[kind + "sharpe"],
                "sortino":    m[kind + "sortino"],
                "max_dd":     m[kind + "max_dd"],
                "calmar":     m[kind + "calmar"],
                "hit_rate":   m[kind + "hit_rate"],
                "skew":       m[kind + "skew"],
                "kurt":       m[kind + "kurt"],
                "type":       "raw" if kind == "raw_" else "exec"
            }
            summary_rows.append(row)

# =========================================================
# SAVE SUMMARY
# =========================================================
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_DIR, "gpt_perf_summary_rx1_ad1.csv")
summary_df.to_csv(summary_path, index=False)

print("\n================= SUMMARY (core columns) =================")
print(summary_df[[
    "instrument","type","fast","slow","scaler",
    "sharpe","sortino","max_dd","calmar","ann_return","ann_vol","hit_rate"
]].to_string(index=False))

print("\n✅ Saved performance summary to:", summary_path)
print("✅ Per-instrument per-speed CSVs saved in:", OUTPUT_DIR)



=== EWMA 2/8 (scaler=10.6) ===
RX1  raw Sharpe=0.053,  Sortino=0.060,  MDD=-63.65%
RX1  exec Sharpe=0.010, Sortino=0.011, MDD=-22.26%
AD1  raw Sharpe=-0.791,  Sortino=-1.000,  MDD=-88.65%
AD1  exec Sharpe=-0.984, Sortino=-1.133, MDD=-21.41%
PORT raw Sharpe=-0.649,  Sortino=-0.829,  MDD=-69.12%
PORT exec Sharpe=-0.606, Sortino=-0.683, MDD=-17.75%

=== EWMA 4/16 (scaler=7.5) ===
RX1  raw Sharpe=-0.567,  Sortino=-0.634,  MDD=-62.93%
RX1  exec Sharpe=-0.345, Sortino=-0.369, MDD=-18.05%
AD1  raw Sharpe=-0.918,  Sortino=-1.179,  MDD=-90.77%
AD1  exec Sharpe=-1.098, Sortino=-1.420, MDD=-18.30%
PORT raw Sharpe=-1.075,  Sortino=-1.423,  MDD=-73.56%
PORT exec Sharpe=-0.969, Sortino=-1.204, MDD=-14.86%

=== EWMA 8/32 (scaler=5.3) ===
RX1  raw Sharpe=-1.251,  Sortino=-1.274,  MDD=-78.05%
RX1  exec Sharpe=-0.996, Sortino=-1.040, MDD=-21.58%
AD1  raw Sharpe=-1.037,  Sortino=-1.303,  MDD=-92.06%
AD1  exec Sharpe=-1.063, Sortino=-1.396, MDD=-16.46%
PORT raw Sharpe=-1.492,  Sortino=-1.945,  MDD=-84.64